# AKOrN Training on MNIST - Configuración Máxima (GPU)

**Objetivo**: Entrenar AKOrN con configuración de máxima precisión en Google Colab GPU

**Método**: 🚀 Ejecutar directamente en Google Colab (https://colab.research.google.com/)

**Parámetros**:
- `--epochs 200`
- `--n 4` (dimensión osciladores)
- `--L 3` (3 capas Kuramoto)
- `--T 5` (5 timesteps por capa)
- `--ch 128` (128 canales base)
- `--batchsize 128`

**Tiempo estimado**: ~18 horas en GPU T4

**Accuracy esperada**: ~99%

---

## 📝 Instrucciones de Uso

### Opción 1: Abrir en Google Colab (Recomendado)
1. Ve a https://colab.research.google.com/
2. File → Open notebook → GitHub
3. Busca: `ACRCris/Proyecto-Inv.-teorica.`
4. Branch: `mac`
5. Selecciona: `codigo/akorn/train_mnist_colab.ipynb`
6. Runtime → Change runtime type → GPU (T4)
7. Ejecuta las celdas en orden (1 → 2 → 3 → ... )

### Opción 2: Desde VSCode con vscode-colab
1. Abrir este notebook en VSCode
2. Click "Select Kernel" → "Connect to Colab"
3. Autenticar con Google
4. Ejecutar celdas secuencialmente

---

## ⚠️ IMPORTANTE - Evitar errores comunes

1. **Ejecuta Runtime → Restart runtime** si encuentras errores de `getcwd`
2. **NO ejecutes la Celda 2 múltiples veces** sin hacer restart primero
3. **Ejecuta las celdas EN ORDEN** (1, 2, 3, 4, ...)
4. **Para entrenamientos largos**: Usa Colab Pro/Pro+ para evitar desconexiones
5. **Mantén la pestaña de Colab abierta** durante el entrenamiento

## 1. Setup Inicial - Verificar GPU

In [1]:
import torch
import os

# Verificar GPU
print("🔍 Verificando GPU...")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA versión: {torch.version.cuda}")
    print(f"Memoria GPU total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ GPU no disponible, usando CPU (muy lento)")

# Configurar para usar toda la GPU
torch.backends.cudnn.benchmark = True
print("\n✅ cuDNN benchmark habilitado")

🔍 Verificando GPU...
CUDA disponible: True
GPU: Tesla T4
CUDA versión: 12.6
Memoria GPU total: 15.83 GB

✅ cuDNN benchmark habilitado


## 2. Clonar Repositorio en Colab

**¿Por qué?** vscode-colab solo sincroniza el notebook, no los archivos del proyecto (train_classification.py, source/, etc.)

**Solución**: Clonar el repositorio de GitHub en el servidor de Colab

In [ ]:
# CELDA 2: Clonar repositorio AKOrN - VERSION ROBUSTA
# Evita errores de "getcwd: cannot access parent directories"

import os
import shutil
from pathlib import Path

print("=" * 70)
print("CELDA 2: Clonar repositorio AKOrN")
print("=" * 70)

# PASO 1: CRITICO - Resetear a /content SIEMPRE
# Esto evita el error "getcwd: cannot access parent directories"
try:
    os.chdir('/content')
    print("[OK] Working directory reseteado a /content")
except Exception as e:
    print(f"[WARN] Error al cambiar a /content: {e}")
    # Fallback con magic command
    %cd /content

print(f"[DEBUG] PWD inicial: {os.getcwd()}\n")

# PASO 2: Definir paths absolutos
repo_name = 'Proyecto-Inv.-teorica.'
repo_path = Path('/content') / repo_name
akorn_path = repo_path / 'codigo' / 'akorn'

# PASO 3: Limpiar clone anterior usando shutil (más robusto que rm -rf)
if repo_path.exists():
    print(f"[LIMPIEZA] Eliminando {repo_name}...")
    try:
        shutil.rmtree(repo_path, ignore_errors=True)
        print("[OK] Limpieza completada")
    except Exception as e:
        print(f"[WARN] shutil.rmtree fallo: {e}")
        print("[FALLBACK] Usando rm -rf...")
        !rm -rf /content/{repo_name}
    
    # Verificar que se eliminó
    if repo_path.exists():
        print(f"[ERROR] No se pudo eliminar {repo_name}")
        print("[SOLUCION] Ejecuta: Runtime -> Restart runtime")
        raise SystemExit("Reinicia runtime y ejecuta esta celda nuevamente")
else:
    print("[INFO] No existe clone anterior (primera ejecucion)")

# PASO 4: Clonar repositorio (SIN --quiet para ver progreso)
print(f"\n[GIT] Clonando repositorio...")
print("Repo: https://github.com/ACRCris/Proyecto-Inv.-teorica..git")
print("Tiempo estimado: 10-20 segundos\n")

# Clonar mostrando progreso
clone_result = !git clone https://github.com/ACRCris/Proyecto-Inv.-teorica..git 2>&1

# Mostrar output del clone
for line in clone_result:
    print(line)

# PASO 5: Verificar que el clone fue exitoso
if not repo_path.exists():
    print("\n" + "!" * 70)
    print("ERROR: git clone fallo")
    print("!" * 70)
    print("\n[DEBUG] Contenido de /content:")
    !ls -la /content
    print("\n[FALLBACK] Intentando clone shallow...")
    !git clone --depth 1 https://github.com/ACRCris/Proyecto-Inv.-teorica..git
    
    if not repo_path.exists():
        print("\n[ERROR CRITICO] No se pudo clonar el repositorio")
        print("Verifica:")
        print("  1. Conectividad de Colab a internet")
        print("  2. Que el repositorio sea publico o tengas acceso")
        raise SystemExit("No se puede continuar sin el repositorio")

print(f"\n[OK] Repositorio clonado en: {repo_path}")

# PASO 6: Verificar estructura antes de cambiar directorio
print(f"\n[DEBUG] Verificando estructura...")
print(f"  Existe {repo_path}? {repo_path.exists()}")
print(f"  Existe {repo_path/'codigo'}? {(repo_path/'codigo').exists()}")
print(f"  Existe {akorn_path}? {akorn_path.exists()}")

if not akorn_path.exists():
    print(f"\n[ERROR] {akorn_path} no existe")
    print(f"[DEBUG] Contenido de {repo_path}:")
    !ls -la {repo_path}
    print(f"[DEBUG] Contenido de {repo_path}/codigo:")
    !ls -la {repo_path}/codigo
    raise SystemExit(f"Path {akorn_path} no encontrado")

# PASO 7: Navegar a codigo/akorn
print(f"\n[NAVEGACION] Cambiando a: {akorn_path}")
%cd {akorn_path}

# Verificar que el cambio funcionó
current_dir = Path.cwd()
print(f"[DEBUG] PWD actual: {current_dir}")

if current_dir != akorn_path:
    print(f"[WARN] PWD esperado: {akorn_path}")
    print(f"[WARN] PWD actual: {current_dir}")
    print("[FIX] Forzando cambio...")
    os.chdir(akorn_path)
    print(f"[OK] PWD corregido: {Path.cwd()}")

# PASO 8: Cambiar a rama 'mac'
print("\n[GIT] Cambiando a rama 'mac' (donde estan las modificaciones)...")
checkout_result = !git checkout mac 2>&1
for line in checkout_result:
    print(line)

# PASO 9: Verificar rama actual
print("\n[GIT] Rama actual:")
branch_result = !git branch --show-current 2>&1
if branch_result:
    print(f"  {branch_result[0]}")

print("\n[GIT] Ultimo commit:")
!git log -1 --oneline

# PASO 10: Verificar archivos CRITICOS usando paths absolutos
print("\n[VERIFICACION] Archivos necesarios:")

archivos_criticos = {
    'train_classification.py': 'Script de entrenamiento',
    'source/models/classification/knet.py': 'Modelo AKOrN',
    'source/evals/classification/adv_attacks.py': 'Ataques adversariales',
    'source/__init__.py': 'Modulo source',
    'requirements.txt': 'Dependencias'
}

missing = []
for archivo_rel, descripcion in archivos_criticos.items():
    # Usar path absoluto para verificación
    archivo_abs = akorn_path / archivo_rel
    
    if archivo_abs.exists():
        size_kb = archivo_abs.stat().st_size / 1024
        print(f"  [OK] {archivo_rel} ({size_kb:.1f} KB)")
    else:
        print(f"  [FALTA] {archivo_rel}")
        missing.append(archivo_rel)

# PASO 11: Preview de train_classification.py
train_script = akorn_path / 'train_classification.py'
if train_script.exists():
    print("\n[PREVIEW] Primeras lineas de train_classification.py:")
    print("-" * 70)
    !head -3 {train_script}
    print("-" * 70)

# PASO 12: Resumen final
print("\n" + "=" * 70)
if missing:
    print(f"ADVERTENCIA: {len(missing)} archivo(s) faltante(s):")
    for f in missing:
        print(f"  - {f}")
    print("\nVerifica que la rama 'mac' tiene estos archivos")
else:
    print("EXITO: Setup completado correctamente")
    print(f"\nUbicacion final: {Path.cwd()}")
    print("\n[SIGUIENTE] Ejecuta Celda 3: Instalar Dependencias")

print("=" * 70)

### 💡 Alternativas (si prefieres no clonar)

**Opción B - Google Drive**:
```python
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Proyecto-Inv.-teorica/codigo/akorn
```
*Requiere: Subir proyecto completo a Drive primero*

**Opción C - Subir archivos manualmente**:
```python
from google.colab import files
uploaded = files.upload()  # Arrastra archivos .py
```
*Tedioso si tienes muchos archivos*

**🎯 Recomendado**: Usar clonar repo (celda anterior) - Rápido y siempre actualizado

## 3. Instalar Dependencias

In [ ]:
# CELDA 3: Instalar Dependencias - VERSION ULTRA ROBUSTA
# Protección contra errores de getcwd con auto-recuperación

import os
from pathlib import Path

print("=" * 70)
print("CELDA 3: Instalar Dependencias")
print("=" * 70)

# PASO 0: CRITICO - Auto-recuperación si getcwd falla
try:
    current = Path.cwd()
    print(f"\n[DEBUG] PWD actual: {current}")
except FileNotFoundError as e:
    print(f"\n[ERROR] getcwd fallo: {e}")
    print("[AUTO-FIX] Reseteando a /content...")
    try:
        os.chdir('/content')
        print(f"[OK] PWD reseteado a: {os.getcwd()}")
    except:
        print("[FALLBACK] Usando magic command...")
        %cd /content
        print(f"[OK] PWD: {os.getcwd()}")

# PASO 1: Verificar y corregir ubicacion ANTES de pip install
akorn_path = Path('/content/Proyecto-Inv.-teorica./codigo/akorn')

print(f"[DEBUG] PWD esperado: {akorn_path}")

# Verificar si el directorio akorn existe
if not akorn_path.exists():
    print(f"\n[ERROR] {akorn_path} no existe")
    print("[SOLUCION] Ejecuta primero la celda anterior (Clonar repositorio)")
    print("\nPasos:")
    print("  1. Runtime → Restart runtime")
    print("  2. Ejecuta celda 1 (Verificar GPU)")
    print("  3. Ejecuta celda 2 (Clonar repositorio)")
    print("  4. Vuelve a ejecutar esta celda")
    raise SystemExit("Directorio akorn no encontrado")

# Si no estamos en el directorio correcto, cambiar
try:
    if Path.cwd() != akorn_path:
        print(f"\n[FIX] Cambiando a {akorn_path}...")
        os.chdir(akorn_path)
        print(f"[OK] PWD corregido: {Path.cwd()}")
except FileNotFoundError:
    # Si Path.cwd() falla, forzar cambio directamente
    print(f"\n[FIX] Forzando cambio a {akorn_path}...")
    os.chdir(akorn_path)
    print(f"[OK] PWD: {os.getcwd()}")

print(f"\n[OK] Ubicacion verificada: {os.getcwd()}\n")

# PASO 2: Instalar PyTorch con CUDA
print("=" * 70)
print("Instalando PyTorch con soporte CUDA...")
print("=" * 70)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# PASO 3: Instalar otras dependencias
print("\n" + "=" * 70)
print("Instalando otras dependencias de AKOrN...")
print("=" * 70)
!pip install -q tensorboard tqdm einops ema-pytorch autoattack

print("\n" + "=" * 70)
print("Dependencias instaladas")
print("=" * 70)

# PASO 4: Verificar versiones instaladas
print("\n[VERIFICACION] Versiones instaladas:")
import torch
import torchvision
print(f"  PyTorch: {torch.__version__}")
print(f"  Torchvision: {torchvision.__version__}")
print(f"  CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

print("\n[SIGUIENTE] Ejecuta la celda siguiente: Descargar MNIST")
print("=" * 70)

## 4. Descargar MNIST

In [ ]:
# CELDA 4: Descargar MNIST - VERSION ULTRA ROBUSTA

import os
from pathlib import Path
import torchvision
from torchvision import transforms

print("=" * 70)
print("CELDA 4: Descargar MNIST")
print("=" * 70)

# PASO 0: Auto-recuperación si getcwd falla
try:
    current = Path.cwd()
    print(f"\n[DEBUG] PWD actual: {current}")
except FileNotFoundError:
    print(f"\n[ERROR] getcwd fallo")
    print("[AUTO-FIX] Reseteando a /content...")
    os.chdir('/content')
    print(f"[OK] PWD: {os.getcwd()}")

# Verificar ubicacion y cambiar si es necesario
akorn_path = Path('/content/Proyecto-Inv.-teorica./codigo/akorn')

if not akorn_path.exists():
    print(f"\n[ERROR] {akorn_path} no existe")
    print("[SOLUCION] Ejecuta las celdas anteriores primero")
    raise SystemExit("Directorio akorn no encontrado")

# Cambiar al directorio correcto
try:
    if Path.cwd() != akorn_path:
        print(f"[FIX] Cambiando a {akorn_path}")
        os.chdir(akorn_path)
except FileNotFoundError:
    print(f"[FIX] Forzando cambio a {akorn_path}")
    os.chdir(akorn_path)

print(f"[OK] PWD: {os.getcwd()}\n")

# Crear directorio de datos
data_dir = Path('./data')
data_dir.mkdir(exist_ok=True)
print(f"[OK] Directorio de datos: {data_dir.absolute()}\n")

# Descargar MNIST
print("Descargando MNIST...")
print("  Train: 60,000 imagenes")
print("  Test: 10,000 imagenes")
print("  Tamano total: ~12 MB\n")

transform = transforms.Compose([transforms.ToTensor()])

trainset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
testset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

print("=" * 70)
print(f"MNIST descargado exitosamente")
print(f"  Train: {len(trainset)} imagenes")
print(f"  Test: {len(testset)} imagenes")
print("=" * 70)

print("\n[SIGUIENTE] Ejecuta la celda siguiente: Configurar Google Drive (opcional)")
print("           O salta a la celda de entrenamiento")
print("=" * 70)

## 5. Configurar Google Drive (Opcional - Para guardar checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Crear directorio para guardar resultados
RESULTS_DIR = '/content/drive/MyDrive/akorn_mnist_maxima'
!mkdir -p {RESULTS_DIR}
print(f"✅ Resultados se guardarán en: {RESULTS_DIR}")

## 6. Iniciar Entrenamiento - Configuración MÁXIMA

**ADVERTENCIA**: Este entrenamiento tomará ~18 horas. Asegúrate de:
1. Estar en Colab Pro/Pro+ para tiempo ilimitado de GPU
2. Tener Google Drive montado para no perder checkpoints
3. Habilitar guardado automático cada 25 epochs

In [ ]:
# Comando de entrenamiento completo
!python train_classification.py mnist_maxima_precision \
    --data mnist \
    --epochs 200 \
    --n 4 \
    --L 3 \
    --T 5 \
    --ch 128 \
    --batchsize 128 \
    --lr 0.0001 \
    --device cuda \
    --checkpoint_every 25 \
    --adveval_freq 10 \
    --pgdeval_freq 50

## 7. Monitoreo Durante Entrenamiento (En otra celda)

Ejecuta esto en paralelo para monitorear el progreso:

In [ ]:
# Ver logs en tiempo real
!tail -f runs/mnist_maxima_precision/events.out.tfevents.*

## 8. Lanzar TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/mnist_maxima_precision

# Alternativamente, usar ngrok para acceso externo:
# !pip install -q pyngrok
# from pyngrok import ngrok
# ngrok.set_auth_token("TU_TOKEN_AQUI")  # Obtén token en ngrok.com
# public_url = ngrok.connect(6006)
# print(f"TensorBoard disponible en: {public_url}")

## 9. Verificar Uso de GPU en Tiempo Real

In [ ]:
# Ejecutar cada 30 segundos para monitorear GPU
import time
import torch

while True:
    if torch.cuda.is_available():
        mem_allocated = torch.cuda.memory_allocated(0) / 1e9
        mem_reserved = torch.cuda.memory_reserved(0) / 1e9
        print(f"GPU Memoria: {mem_allocated:.2f} GB usada / {mem_reserved:.2f} GB reservada")
    time.sleep(30)
    
# Detener con: Runtime -> Interrupt execution

## 10. Guardar Checkpoints en Drive (Automático cada 25 epochs)

In [ ]:
# Copiar checkpoints a Drive automáticamente
import shutil
import glob

checkpoints = glob.glob('runs/mnist_maxima_precision/checkpoint_*.pth')
for ckpt in checkpoints:
    dest = f"{RESULTS_DIR}/{os.path.basename(ckpt)}"
    shutil.copy(ckpt, dest)
    print(f"✅ Copiado: {ckpt} → {dest}")

# Copiar también EMA
ema_files = glob.glob('runs/mnist_maxima_precision/ema_*.pth')
for ema in ema_files:
    dest = f"{RESULTS_DIR}/{os.path.basename(ema)}"
    shutil.copy(ema, dest)
    print(f"✅ Copiado EMA: {ema} → {dest}")

## 11. Cargar Checkpoint y Evaluar (Después del entrenamiento)

In [ ]:
import torch
from source.models.classification.knet import AKOrN

# Cargar modelo desde checkpoint
device = torch.device('cuda')

model = AKOrN(
    n=4,
    ch=128,
    out_classes=10,
    L=3,
    T=5,
    J="conv",
    ksizes=[9, 7, 5],
    ro_ksize=3,
    ro_N=2,
    norm="bn",
    c_norm="gn",
    gamma=1.0,
    use_omega=True,
    init_omg=1.0,
    global_omg=True,
    learn_omg=True,
    ensemble=1,
).to(device)

# Cargar pesos del checkpoint final
checkpoint = torch.load('runs/mnist_maxima_precision/checkpoint_epoch_200.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print("✅ Modelo cargado desde checkpoint epoch 200")

# Evaluar
model.eval()
correct = 0
total = 0

testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False)

with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"\n🎯 Accuracy Final: {accuracy:.2f}%")

## 12. Análisis de Resultados

### Accuracy por Epoch (esperado):
- Epoch 10: ~87%
- Epoch 50: ~95%
- Epoch 100: ~97%
- Epoch 150: ~98%
- Epoch 200: ~99%

### Robustez Adversarial (esperado):
- Clean: ~99%
- Random Noise: ~96%
- FGSM (ε=8/255): ~90%
- PGD (ε=8/255): ~80%

## 13. Descargar Resultados Completos

In [ ]:
# Comprimir todos los resultados
!zip -r mnist_maxima_results.zip runs/mnist_maxima_precision/

# Descargar (click derecho en el archivo en el explorador de archivos)
from google.colab import files
files.download('mnist_maxima_results.zip')

print("✅ Archivo listo para descargar")

## 14. Notas Importantes

### Tiempo de Ejecución:
- GPU T4 (Colab Free): ~18-20 horas
- GPU V100 (Colab Pro+): ~10-12 horas
- GPU A100 (Colab Pro+): ~6-8 horas

### Memoria GPU Requerida:
- Batch size 128 con ch=128, L=3, T=5: ~8-10 GB
- Si encuentras OOM (Out of Memory):
  - Reducir `--batchsize 64`
  - O reducir `--ch 96`

### Guardado de Checkpoints:
- Cada 25 epochs (configurable con `--checkpoint_every`)
- Modelo base + EMA guardados por separado
- Total ~2 GB por checkpoint

### Reanudar Entrenamiento:
Si se interrumpe, añadir `--finetune runs/mnist_maxima_precision/checkpoint_epoch_XXX.pth`